[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_01/01_algebra_vectorial_coordenadas_gradiente.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 1 — Álgebra vectorial, sistemas coordenados y gradiente

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 1**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted será capaz de:

1. Expresar un mismo vector en coordenadas cilíndricas y cartesianas, y
   explicar por qué sus componentes cambian aunque el vector sea el mismo.
2. Calcular magnitud, dirección y producto cruz de un campo vectorial, y
   verificar que la magnitud es invariante ante el cambio de base.
3. Obtener el campo eléctrico a partir de un potencial escalar mediante
   $\mathbf{E} = -\nabla V$, primero de forma simbólica y luego numérica.
4. Interpretar un mapa de contorno del potencial y relacionar la densidad
   de las curvas de nivel con la intensidad del campo.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "transformaciones_coordenadas.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from transformaciones_coordenadas import cilindricas_a_cartesianas
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. Fundamentos teóricos y matemáticos

### 2.1 Bases ortogonales y cambio de sistema coordenado

Un vector es un objeto geométrico: existe con independencia del sistema de
coordenadas que usemos para describirlo. Lo que sí depende del sistema son
sus *componentes*.

En coordenadas cartesianas la base $\{\mathbf{a}_x, \mathbf{a}_y,
\mathbf{a}_z\}$ es fija: apunta en la misma dirección en todo punto del
espacio. En coordenadas cilíndricas la base $\{\mathbf{a}_\rho,
\mathbf{a}_\phi, \mathbf{a}_z\}$ **gira con el punto de observación**: en
$\phi = 0$ el versor $\mathbf{a}_\rho$ coincide con $\mathbf{a}_x$, y en
$\phi = 90^\circ$ coincide con $\mathbf{a}_y$.

Por eso, para convertir componentes cilíndricas a cartesianas hay que
indicar en qué ángulo $\phi$ se está evaluando el campo.

### 2.2 El gradiente y el potencial electrostático

En electrostática el campo eléctrico es conservativo, $\nabla \times
\mathbf{E} = 0$, lo que permite escribirlo como el gradiente de un
potencial escalar. El signo negativo es una convención física: el campo
apunta hacia donde el potencial *disminuye*, de modo que una carga positiva
libre se mueve espontáneamente hacia potenciales menores.

Trabajar con $V$ en lugar de $\mathbf{E}$ simplifica los problemas: $V$ es
una sola función escalar, mientras que $\mathbf{E}$ son tres funciones.

## 3. Ecuaciones relevantes

**Versores cilíndricos en función de los cartesianos:**

$$
\mathbf{a}_\rho = \cos\phi\,\mathbf{a}_x + \sin\phi\,\mathbf{a}_y,
\qquad
\mathbf{a}_\phi = -\sin\phi\,\mathbf{a}_x + \cos\phi\,\mathbf{a}_y .
$$

**Cambio de base de un vector $\mathbf{E} = E_\rho \mathbf{a}_\rho +
E_\phi \mathbf{a}_\phi + E_z \mathbf{a}_z$:**

$$
\begin{aligned}
E_x &= E_\rho \cos\phi - E_\phi \sin\phi, \\
E_y &= E_\rho \sin\phi + E_\phi \cos\phi, \\
E_z &= E_z .
\end{aligned}
$$

**Magnitud y dirección en el plano $xy$:**

$$
|\mathbf{E}| = \sqrt{E_x^2 + E_y^2 + E_z^2},
\qquad
\theta_{xy} = \arctan\!\left(\frac{E_y}{E_x}\right).
$$

**Producto cruz con el versor axial:**

$$
\mathbf{E} \times \mathbf{a}_z =
\begin{vmatrix}
\mathbf{a}_x & \mathbf{a}_y & \mathbf{a}_z \\
E_x & E_y & E_z \\
0 & 0 & 1
\end{vmatrix}
= E_y\,\mathbf{a}_x - E_x\,\mathbf{a}_y .
$$

**Gradiente en coordenadas cartesianas y campo eléctrico:**

$$
\nabla V = \frac{\partial V}{\partial x}\mathbf{a}_x
         + \frac{\partial V}{\partial y}\mathbf{a}_y
         + \frac{\partial V}{\partial z}\mathbf{a}_z,
\qquad
\mathbf{E} = -\nabla V .
$$

**Potencial de trabajo de este notebook:**

$$
V(x,z) = V_0\,e^{-\alpha z}\cos\!\left(\frac{\pi x}{2}\right),
\qquad V_0 = 20.0~\text{V},\quad \alpha = 0.5~\text{m}^{-1}.
$$

Derivando y aplicando $\mathbf{E} = -\nabla V$:

$$
\mathbf{E}(x,z) =
\frac{\pi V_0}{2} e^{-\alpha z}\sin\!\left(\frac{\pi x}{2}\right)\mathbf{a}_x
+ \alpha V_0 e^{-\alpha z}\cos\!\left(\frac{\pi x}{2}\right)\mathbf{a}_z .
$$

## 4. Explicación física

**Sobre el cambio de base.** La magnitud $|\mathbf{E}|$ no cambia al pasar
de cilíndricas a cartesianas: una rotación de la base no puede alterar la
longitud del vector. Esto es una comprobación gratuita y muy útil: si su
resultado numérico cambia la magnitud, el cambio de base está mal hecho.

**Sobre el producto cruz $\mathbf{E} \times \mathbf{a}_z$.** El resultado es
perpendicular a ambos: es $\mathbf{E}$ girado $90^\circ$ en el plano $xy$ y
sin la componente axial. Esta operación aparecerá una y otra vez en el
curso, porque el vector de Poynting $\mathbf{E} \times \mathbf{H}$ tiene la
misma estructura.

**Sobre el potencial.** La forma $e^{-\alpha z}\cos(\pi x/2)$ describe una
distribución que oscila en $x$ y se atenúa al alejarse en $z$. El campo
eléctrico apunta siempre en la dirección de máxima caída de $V$ y es
perpendicular a las curvas de nivel del potencial. Donde las curvas de nivel
se aprietan, el campo es intenso; donde se separan, es débil.

## 5. Parámetros modificables

Modifique **solo** esta celda y vuelva a ejecutar el notebook completo.

In [ ]:
# --- Problema 1: vector en coordenadas cilíndricas ---
phi_grados = 30.0   # ángulo de observación [grados]
E_rho = 5.0         # componente radial [V/m]
E_phi = 2.0         # componente azimutal [V/m]

# --- Problema 2: potencial y punto de evaluación ---
V0 = 20.0           # amplitud del potencial [V]
alpha = 0.5         # tasa de atenuación en z [1/m]
x_p = 0.5           # coordenada x del punto P [m]
z_p = 1.0           # coordenada z del punto P [m]

## 6. Implementación del código

### 6.1 Problema 1 — cambio de base

La función `cilindricas_a_cartesianas` vive en `src/transformaciones_coordenadas.py`.

In [ ]:
phi = np.deg2rad(phi_grados)
E_cartesiano = cilindricas_a_cartesianas(E_rho, E_phi, 0.0, phi)

magnitud_E = np.linalg.norm(E_cartesiano)
angulo_xy = np.rad2deg(np.arctan2(E_cartesiano[1], E_cartesiano[0]))
producto_cruz = np.cross(E_cartesiano, [0.0, 0.0, 1.0])

### 6.2 Problema 2 — el gradiente, primero en forma simbólica

En lugar de escribir a mano las derivadas parciales, las obtenemos con
SymPy y comprobamos que coinciden con la expresión analítica de la
sección 3. Así el resultado queda verificado, no supuesto.

In [ ]:
x_sim, z_sim = sp.symbols("x z", real=True)
V_simbolico = V0 * sp.exp(-alpha * z_sim) * sp.cos(sp.pi * x_sim / 2)

E_x_simbolico = -sp.diff(V_simbolico, x_sim)
E_z_simbolico = -sp.diff(V_simbolico, z_sim)

print("V(x, z)   =", sp.simplify(V_simbolico))
print("E_x(x, z) =", sp.simplify(E_x_simbolico))
print("E_z(x, z) =", sp.simplify(E_z_simbolico))

Ahora convertimos las expresiones simbólicas en funciones numéricas de
NumPy y verificamos que reproducen la forma cerrada de la sección 3.

In [ ]:
potencial = sp.lambdify((x_sim, z_sim), V_simbolico, "numpy")
campo_x = sp.lambdify((x_sim, z_sim), E_x_simbolico, "numpy")
campo_z = sp.lambdify((x_sim, z_sim), E_z_simbolico, "numpy")


def campo_electrico(x, z):
    """Vector E = -grad(V) evaluado en (x, z), con E_y = 0 por simetría."""
    return np.array([campo_x(x, z), 0.0, campo_z(x, z)])


def campo_electrico_analitico(x, z):
    """Forma cerrada obtenida a mano en la sección 3."""
    componente_x = (np.pi * V0 / 2.0) * np.exp(-alpha * z) * np.sin(np.pi * x / 2.0)
    componente_z = alpha * V0 * np.exp(-alpha * z) * np.cos(np.pi * x / 2.0)
    return np.array([componente_x, 0.0, componente_z])


E_punto = campo_electrico(x_p, z_p)
E_punto_analitico = campo_electrico_analitico(x_p, z_p)

assert np.allclose(E_punto, E_punto_analitico), "La derivada simbólica y la analítica difieren"
print("La derivada simbólica coincide con la forma cerrada.")

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Componente x del campo", "E_x", E_cartesiano[0], "V/m"),
        ("Componente y del campo", "E_y", E_cartesiano[1], "V/m"),
        ("Componente z del campo", "E_z", E_cartesiano[2], "V/m"),
        ("Magnitud del campo", "|E|", magnitud_E, "V/m"),
        ("Ángulo en el plano xy", "theta_xy", angulo_xy, "grados"),
        ("Producto cruz, componente x", "(E x a_z)_x", producto_cruz[0], "V/m"),
        ("Producto cruz, componente y", "(E x a_z)_y", producto_cruz[1], "V/m"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Potencial en P", "V(P)", float(potencial(x_p, z_p)), "V"),
        ("Campo en P, componente x", "E_x(P)", E_punto[0], "V/m"),
        ("Campo en P, componente z", "E_z(P)", E_punto[2], "V/m"),
        ("Magnitud del campo en P", "|E(P)|", float(np.linalg.norm(E_punto)), "V/m"),
    ]
)

Una comprobación rápida: la magnitud en coordenadas cilíndricas es
$\sqrt{E_\rho^2 + E_\phi^2}$. Debe coincidir con $|\mathbf{E}|$ de la
tabla anterior.

In [ ]:
magnitud_cilindrica = np.hypot(E_rho, E_phi)
print(f"|E| en cilíndricas = {magnitud_cilindrica:.6f} V/m")
print(f"|E| en cartesianas = {magnitud_E:.6f} V/m")
print(f"Diferencia = {abs(magnitud_cilindrica - magnitud_E):.3e} V/m")

## 8. Visualización

El mapa de color muestra $V(x,z)$; las flechas muestran $\mathbf{E} =
-\nabla V$ en la misma región. Observe que las flechas cruzan las
transiciones de color en ángulo recto.

In [ ]:
x = np.linspace(-2.0, 2.0, 241)
z = np.linspace(0.0, 3.0, 181)
X, Z = np.meshgrid(x, z)
V = potencial(X, Z)

fig, ax = plt.subplots(figsize=(7.0, 4.5))
niveles = np.linspace(V.min(), V.max(), 17)
mapa = ax.contourf(X, Z, V, levels=niveles, cmap="coolwarm")
fig.colorbar(mapa, ax=ax, label="V (V)")

paso = 20
ax.quiver(
    X[::paso, ::paso],
    Z[::paso, ::paso],
    campo_x(X[::paso, ::paso], Z[::paso, ::paso]),
    campo_z(X[::paso, ::paso], Z[::paso, ::paso]),
    color="black",
    alpha=0.7,
    width=0.003,
)
ax.scatter([x_p], [z_p], color="black", zorder=5, label="P")
ax.set_xlabel("x (m)")
ax.set_ylabel("z (m)")
ax.set_title("Potencial V(x, z) y campo E = -grad(V)")
ax.grid(False)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Interpretación física de los resultados

**Cambio de base.** Las componentes cartesianas $(E_x, E_y)$ no se parecen
a $(E_\rho, E_\phi)$, pero la magnitud es idéntica: la base cilíndrica está
girada $30^\circ$ respecto de la cartesiana en el punto de evaluación, y una
rotación no cambia longitudes. El ángulo resultante en el plano $xy$ es
exactamente $\phi + \arctan(E_\phi/E_\rho)$.

**Producto cruz.** $\mathbf{E}\times\mathbf{a}_z$ tiene la misma magnitud
que la proyección de $\mathbf{E}$ sobre el plano $xy$ y está girado
$-90^\circ$ respecto de ella. La componente $z$ del resultado es nula, como
debe ser: el producto cruz es perpendicular a $\mathbf{a}_z$.

**Campo y potencial.** En el punto $P$ el campo tiene una componente $x$
unas tres veces mayor que la componente $z$. La razón es geométrica: en
$x = 0.5$ m la función $\cos(\pi x/2)$ está cambiando rápido, mientras que
la atenuación $e^{-\alpha z}$ con $\alpha = 0.5~\text{m}^{-1}$ es lenta. El
gradiente es grande donde la función varía rápido.

**Lectura del gráfico.** Las flechas nacen en las zonas rojas (potencial
alto) y mueren en las azules (potencial bajo), y en ningún punto son
tangentes a una curva de nivel. Esa perpendicularidad es la definición
geométrica del gradiente, no una coincidencia numérica.

## 10. Ejercicios para experimentar

            Responda modificando **solo** la celda de parámetros de la sección 5.

            1. Ponga `phi_grados = 0.0`. ¿Qué relación hay ahora entre $(E_\rho,
               E_\phi)$ y $(E_x, E_y)$? Explique el resultado usando las expresiones de
               los versores.
            2. Con `phi_grados = 90.0`, prediga $(E_x, E_y)$ **antes** de ejecutar.
               Compare con el resultado.
            3. Duplique `alpha` a `1.0`. ¿Cómo cambia $E_z(P)$? ¿Y $E_x(P)$? Justifique
               cuál de las dos componentes depende de `alpha` mirando las ecuaciones de
               la sección 3.
            4. Mueva el punto a `x_p = 0.0`. ¿Por qué se anula $E_x$? Relaciónelo con la
               pendiente de $\cos(\pi x/2)$ en ese punto.
            5. Mueva el punto a `x_p = 1.0`. Ahora se anula $E_z$. ¿Qué tienen en común
               los puntos donde una componente del campo se anula y las curvas de nivel
               del gráfico?
            6. Triplique `V0`. ¿Cambia la *dirección* del campo en $P$? ¿Y su magnitud?
               ¿Qué le dice esto sobre la linealidad de $\mathbf{E} = -\nabla V$?